# Digital Earth Australia Mosaic Metadata Command Line Interfaces (CLI)

This notebook demonstrates how to call the DEA Mosaic Metadata workflows via Command Line Interface utilities. 
This allows you to generate ODC STAC and YAML metadata and thumbnails for DEA mosaic products, using a single line of code.

## Getting started
Set working directory to top level of repo to ensure links work correctly:



In [ ]:
cd ../..

Install the latest version of `dea-intertidal`:


In [ ]:
%pip install -e .

In [ ]:
%load_ext autoreload
%autoreload 2

import rasterio
print(rasterio.__gdal_version__)
import datacube
print(datacube.__version__)


## Setup


### Set analysis parameters
Sets the required parameters year, product and version. 

In [ ]:
year = "2024"
product = "ga_s2ls_intertidal_cyear_3"
version = "2-1-0"

## DEA Mosaic Metadata CLI
This CLI allows you to generate the following DEA Metadata outputs with a single command line call:
* **ODC STAC metadata**
* **ODC YAML metdata**
* **thumbnail**

Running `--help` shows all the CLI parameters that can be used to customise the analysis:

In [ ]:
!python -m intertidal.metadata_generator generate --help

Run DEA Mosaic Metadata CLI for a single epoch. This will generate DEA Mosaic Metadata for a single year for a single product:

In [ ]:
%%time

!python -m intertidal.metadata_generator generate \
    --year {year} \
    --product {product} \
    --version {version} \
    --verbose

### Validation

Use the DEA Mosaic Metadata CLI to validate a local metadata file:

In [ ]:
file_stac = "metadata/ga_s2ls_intertidal_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2ls_intertidal_cyear_3_mosaic_2024--P1Y.stac-item.json"
file_yaml = "metadata/ga_s2ls_intertidal_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2ls_intertidal_cyear_3_mosaic_2024--P1Y.odc-metadata.yaml"

!python -m intertidal.metadata_generator validate {file_stac}
!python -m intertidal.metadata_generator validate {file_yaml}

Use the DEA Mosaic Metadata CLI to validate a metadata file on S3:

In [ ]:
file_stac = "https://data.dev.dea.ga.gov.au/derivative/ga_s2ls_intertidal_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2ls_intertidal_cyear_3_mosaic_2024--P1Y.stac-item.json"
file_yaml = "https://data.dev.dea.ga.gov.au/derivative/ga_s2ls_intertidal_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2ls_intertidal_cyear_3_mosaic_2024--P1Y.odc-metadata.yaml"

!python -m intertidal.metadata_generator validate {file_stac}
!python -m intertidal.metadata_generator validate {file_yaml}

### Previewing paths
Use DEA Mosaic Metadata CLI to show the paths it will use:

In [ ]:
!python -m intertidal.metadata_generator show-paths \
    --year 2024 \
    --product ga_s2ls_intertidal_cyear_3 \
    --version 2-1-0 \
    --prod

### Run DEA Mosaic Metadata for Tidal Composites for a single epoch
This will generate metadata for DEA Intertidal Composites mosaics:

In [ ]:
%%time
year = "2024"
product = "ga_s2_tidal_composites_cyear_3"
version = "1-1-0"

# Run DEA Mosaic Metadata CLI for a single epoch.
# This will generate DEA Mosaic Metadata for a single year for a single product
!python -m intertidal.metadata_generator generate  \
    --year {year} \
    --product {product} \
    --version {version} \
    --verbose

### Run DEA Mosaic Metadata for Coastal Ecosystems for a single epoch
This will generate metadata for DEA Coastal Ecosystems mosaics:

In [ ]:
%%time
year = "2021"
product = "ga_s2_coastalecosystems_cyear_3_v1"
version = "1-0-0"

# Run DEA Mosaic Metadata CLI for a single epoch.
# This will generate DEA Mosaic Metadata for a single year for a single product
!python -m intertidal.metadata_generator generate  \
    --year {year} \
    --product {product} \
    --version {version} \
    --prod \
    --naming-conventions dea \
    --tile-dir dea-public-data-dev/derivative/dea_coastalecosystems \
    --verbose

## Test loading data via `odc-stac`

DEA Tidal Composites:

In [ ]:
from pystac import Item
import odc.stac
import rasterio
import odc.geo.xr

# Build a query with the parameters above
geometry = [150.4, -22.4, 150.8, -22.7]
path = "https://dea-public-data-dev.s3-ap-southeast-2.amazonaws.com/derivative/ga_s2_tidal_composites_cyear_3/1-1-0/continental_mosaics/2024--P1Y/ga_s2_tidal_composites_cyear_3_2024--P1Y_final.stac-item.json"
# path = "metadata/ga_s2_tidal_composites_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2_tidal_composites_cyear_3_mosaic_2024--P1Y.stac-item.json"
items = [Item.from_file(path)]

# data = odc.stac.load(items, chunks={})
ds = odc.stac.load(
    items,
    bands=["low_red"],
    crs="utm",
    resolution=10,
    # groupby="solar_day",
    bbox=geometry,
)

ds.low_red.plot()

DEA Intertidal:

In [ ]:
from pystac import Item
import odc.stac
import rasterio
import odc.geo.xr

# Build a query with the parameters above
geometry = [150.4, -22.4, 150.8, -22.7]
path = "https://data.dev.dea.ga.gov.au/derivative/ga_s2ls_intertidal_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2ls_intertidal_cyear_3_mosaic_2024--P1Y.stac-item.json"
# path = 'metadata/ga_s2ls_intertidal_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2ls_intertidal_cyear_3_mosaic_2024--P1Y.stac-item.json'

items = [Item.from_file(path)]
ds = odc.stac.load(
    items,
    bands=["elevation"],
    crs="utm",
    resolution=10,
    # groupby="solar_day",
    bbox=geometry,
)

ds.elevation.plot()

DEA Coastal Ecosystems:

In [ ]:
from pystac import Item
import odc.stac
import rasterio
import odc.geo.xr

# Build a query with the parameters above
geometry = [150.4, -22.4, 150.8, -22.7]
path = "https://data.dea.ga.gov.au/derivative/ga_s2_coastalecosystems_cyear_3_v1/AU/2021--P1Y/ga_s2_coastalecosystems_cyear_3_v1-0-0_AU_2021--P1Y_final.stac-item.json"
path = "metadata/ga_s2_coastalecosystems_cyear_3_v1/AU/2021--P1Y/ga_s2_coastalecosystems_cyear_3_v1-0-0_AU_2021--P1Y_final.stac-item.json"
# path = 'metadata/ga_s2_tidal_composites_cyear_3/2-1-0/continental_mosaics/2024--P1Y/ga_s2_tidal_composites_cyear_3_mosaic_2024--P1Y.stac-item.json'
# path = '../..metadata/ga_s2_coastalecosystems_cyear_3_v1-0-0_AU_2022--P1Y_final.stac-item.json'

items = [Item.from_file(path)]
ds = odc.stac.load(
    items,
    bands=[
        "classification",
        # "mangrove_prob",
        # "saltmarsh_prob",
        # "saltflat_prob",
        # "seagrass_prob",
        # "qa_count_clear",
        # "qa_coastal_connectivity",
    ],
    crs="utm",
    resolution=10,
    # groupby="solar_day",
    bbox=geometry,
)

ds.classification.plot()